<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Import Libraries</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Import Libraries
    </h1>
</div>


In [25]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from config.spark_config import SparkConfig
from config.io_config import *
from app.platform_app import PlatformApp
from transformation.gold_loader import *
from utils.logger import LoggerConfig
from transformation.bronze_ingestion import *
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from utils.data_quality import *
from utils.data_cleaning import *
from utils.utils import *

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Set up</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Set up
    </h1>
</div>


In [26]:
logger = LoggerConfig().setup_logger(log_dir=None)
spark = SparkConfig.create_spark(app_name="FMCG Domain", logger=logger, use_databricks=True)
app = PlatformApp(spark=spark, logger=logger, catalog_name="fmcg_domain")

2026-03-19 23:18:30 | INFO     | Logging configured: level=DEBUG, format=colored
2026-03-19 23:18:30 | INFO     | Connected to Databricks via Spark Connect.
2026-03-19 23:18:30 | INFO     | Initializing Data Platform...
2026-03-19 23:18:30 | INFO     | Spark session initialized


<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Bronze</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Bronze
    </h1>
</div>


In [27]:
# Bronze Ingestion
logger.info("Ingesting data into Bronze layer...")
upload_file_to_bronze(spark=spark, logger=logger, entity="products",
                      path_data=S3_BASE_PATH, path_cp=CP_PATH_BRONZE,
                      name_catalog=app.catalog_name)
logger.info("Bronze ingestion completed.")
logger.info("*" * 80)

2026-03-19 23:18:30 | INFO     | Ingesting data into Bronze layer...
2026-03-19 23:18:30 | INFO     | Starting Bronze ingestion for entity: Products
2026-03-19 23:18:30 | INFO     | Uploading file: s3://00-sportsbar-dp/products
2026-03-19 23:18:37 | INFO     | Products: Schema inferred successfully
2026-03-19 23:18:44 | INFO     | Completed Bronze ingestion for entity: Products
+--------------------------------------------+----------+-----------------+-----------------------+------------+---------+----------------------+
|product_name                                |product_id|category         |read_timestamp         |file_name   |file_size|file_modification_time|
+--------------------------------------------+----------+-----------------+-----------------------+------------+---------+----------------------+
|SportsBar Energy Bar Choco Fudge (60g)      |25891101  |energy bars      |2026-03-19 12:29:55.161|products.csv|1388     |2026-03-05 13:07:43   |
|SportsBar Energy Bar Choco Fudge (

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Silver</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Silver
    </h1>
</div>


In [28]:
df_bronze = spark.sql(f"SELECT * FROM {app.catalog_name}.bronze.products")
df_bronze.show(truncate=False)

+-------------------------------------------------+----------+-----------------+-----------------------+------------+---------+----------------------+
|product_name                                     |product_id|category         |read_timestamp         |file_name   |file_size|file_modification_time|
+-------------------------------------------------+----------+-----------------+-----------------------+------------+---------+----------------------+
|SportsBar Energy Bar Choco Fudge (60g)           |25891101  |energy bars      |2026-03-19 12:29:55.161|products.csv|1388     |2026-03-05 13:07:43   |
|SportsBar Energy Bar Choco Fudge (40g)           |25891102  |energy bars      |2026-03-19 12:29:55.161|products.csv|1388     |2026-03-05 13:07:43   |
|SportsBar Energy Bar Choco Fudge (25g)           |25891103  |energy bars      |2026-03-19 12:29:55.161|products.csv|1388     |2026-03-05 13:07:43   |
|SportsBar Protien Bar Peanut Crunch (45g)        |25891201  |protien bars     |2026-03-19 12:

## Transformations

### Duplicates

In [29]:
df_duplicates = check_duplicate(df=df_bronze, logger=logger)

2026-03-19 23:18:47 | WARNING  | 2 duplicate rows detected (10.00%).
2026-03-19 23:18:47 | WARNING  | Rows affected: 2 out of 20.


In [30]:
df_silver = dedup(df=df_bronze, dedup_cols=["product_id"], 
                  cdc="read_timestamp", logger=logger)

2026-03-19 23:18:47 | INFO     | Starting deduplication
2026-03-19 23:18:47 | INFO     | Dedup columns: ['product_id'] | CDC column: read_timestamp
2026-03-19 23:18:48 | INFO     | Input row count: 20
2026-03-19 23:18:49 | INFO     | Output row count after dedup: 18
2026-03-19 23:18:49 | INFO     | Removed duplicate rows: 2
2026-03-19 23:18:49 | INFO     | Deduplication completed


### Check NULL

In [31]:
# check null
check_null(df = df_silver, logger=logger)

2026-03-19 23:18:51 | INFO     | No missing values detected in 18 rows.


### Trim spaces in customer name

In [32]:
# remove those trim values
df_silver = clean_dataframe(df=df_silver)
df_silver.show(truncate=False)

+-------------------------------------------------+----------+-----------------+-----------------------+------------+---------+----------------------+
|product_name                                     |product_id|category         |read_timestamp         |file_name   |file_size|file_modification_time|
+-------------------------------------------------+----------+-----------------+-----------------------+------------+---------+----------------------+
|SportsBar Energy Bar Choco Fudge (60g)           |25891101  |energy bars      |2026-03-19 12:29:55.161|products.csv|1388     |2026-03-05 13:07:43   |
|SportsBar Energy Bar Choco Fudge (40g)           |25891102  |energy bars      |2026-03-19 12:29:55.161|products.csv|1388     |2026-03-05 13:07:43   |
|SportsBar Energy Bar Choco Fudge (25g)           |25891103  |energy bars      |2026-03-19 12:29:55.161|products.csv|1388     |2026-03-05 13:07:43   |
|SportsBar Protien Bar Peanut Crunch (45g)        |25891201  |protien bars     |2026-03-19 12:

### Fix Title-Casing Issue

In [33]:
# sanity check
df_silver.select("category").distinct().show(truncate=False)

+-----------------+
|category         |
+-----------------+
|energy bars      |
|protien bars     |
|granola & cereals|
|recovery dairy   |
|healthy snacks   |
|electrolyte mix  |
+-----------------+



In [34]:
# Title case fix
df_silver = df_silver.withColumn(
    "category",
    F.when(F.col("category").isNull(), None)
     .otherwise(F.initcap("category"))
)

# sanity check
df_silver.select("category").distinct().show(truncate=False)

+-----------------+
|category         |
+-----------------+
|Energy Bars      |
|Protien Bars     |
|Granola & Cereals|
|Recovery Dairy   |
|Healthy Snacks   |
|Electrolyte Mix  |
+-----------------+



### Fix Spelling Mistake for `Protien`

In [35]:
df_silver.show(truncate=False)

+-------------------------------------------------+----------+-----------------+-----------------------+------------+---------+----------------------+
|product_name                                     |product_id|category         |read_timestamp         |file_name   |file_size|file_modification_time|
+-------------------------------------------------+----------+-----------------+-----------------------+------------+---------+----------------------+
|SportsBar Energy Bar Choco Fudge (60g)           |25891101  |Energy Bars      |2026-03-19 12:29:55.161|products.csv|1388     |2026-03-05 13:07:43   |
|SportsBar Energy Bar Choco Fudge (40g)           |25891102  |Energy Bars      |2026-03-19 12:29:55.161|products.csv|1388     |2026-03-05 13:07:43   |
|SportsBar Energy Bar Choco Fudge (25g)           |25891103  |Energy Bars      |2026-03-19 12:29:55.161|products.csv|1388     |2026-03-05 13:07:43   |
|SportsBar Protien Bar Peanut Crunch (45g)        |25891201  |Protien Bars     |2026-03-19 12:

In [36]:
# Replace 'protien' → 'protein' in both product_name and category
df_silver = (
    df_silver
    .withColumn("product_name", F.regexp_replace(F.col("product_name"), "(?i)Protien", "Protein"))
    .withColumn("category", F.regexp_replace(F.col("category"), "(?i)Protien", "Protein"))
)

df_silver.show(truncate=False)

+-------------------------------------------------+----------+-----------------+-----------------------+------------+---------+----------------------+
|product_name                                     |product_id|category         |read_timestamp         |file_name   |file_size|file_modification_time|
+-------------------------------------------------+----------+-----------------+-----------------------+------------+---------+----------------------+
|SportsBar Energy Bar Choco Fudge (60g)           |25891101  |Energy Bars      |2026-03-19 12:29:55.161|products.csv|1388     |2026-03-05 13:07:43   |
|SportsBar Energy Bar Choco Fudge (40g)           |25891102  |Energy Bars      |2026-03-19 12:29:55.161|products.csv|1388     |2026-03-05 13:07:43   |
|SportsBar Energy Bar Choco Fudge (25g)           |25891103  |Energy Bars      |2026-03-19 12:29:55.161|products.csv|1388     |2026-03-05 13:07:43   |
|SportsBar Protein Bar Peanut Crunch (45g)        |25891201  |Protein Bars     |2026-03-19 12:

### Standardizing Customer Attributes to Match Parent Company Data Model

In [37]:
# Sanity check
df_silver.select("category").distinct().show()

+-----------------+
|         category|
+-----------------+
|      Energy Bars|
|     Protein Bars|
|Granola & Cereals|
|   Recovery Dairy|
|   Healthy Snacks|
|  Electrolyte Mix|
+-----------------+



In [38]:
# 1: Add division column
df_silver = (
    df_silver
    .withColumn(
        "division",
        F.when(F.col("category") == "Energy Bars",          "Nutrition Bars")
         .when(F.col("category") == "Protein Bars",         "Nutrition Bars")
         .when(F.col("category") == "Granola & Cereals",    "Breakfast Foods")
         .when(F.col("category") == "Recovery Dairy",       "Dairy & Recovery")
         .when(F.col("category") == "Healthy Snacks",       "Healthy Snacks")
         .when(F.col("category") == "Electrolyte Mix",      "Hydration & Electrolytes")
         .otherwise("Other")
    )
)

# 2: Variant column
df_silver = df_silver.withColumn(
    "variant",
    F.regexp_extract(F.col("product_name"), r"\((.*)\)", 1)
)

# 3: Create new column: product_code
# Invalid product_ids are replaced with a fallback value to avoid losing fact records
# and ensure downstream joins remain consistent
df_silver = (
    df_silver
    # 1. Generate deterministic product_code from product_name
    .withColumn("product_code", F.sha2(F.col("product_name").cast("string"), 256))

    # 2. Clean product_id: keep only numeric IDs, else set to 999999
    .withColumn(
        "product_id",
        F.when(
            F.col("product_id").cast("string").rlike("^[0-9]+$"),
            F.col("product_id").cast("string")
        ).otherwise(F.lit(999999).cast("string"))
    )
    # 3. Rename product_name → product
    .withColumnRenamed("product_name", "product")
)

df_silver.show(truncate=False)

+-------------------------------------------------+----------+-----------------+-----------------------+------------+---------+----------------------+------------------------+----------+----------------------------------------------------------------+
|product                                          |product_id|category         |read_timestamp         |file_name   |file_size|file_modification_time|division                |variant   |product_code                                                    |
+-------------------------------------------------+----------+-----------------+-----------------------+------------+---------+----------------------+------------------------+----------+----------------------------------------------------------------+
|SportsBar Energy Bar Choco Fudge (60g)           |25891101  |Energy Bars      |2026-03-19 12:29:55.161|products.csv|1388     |2026-03-05 13:07:43   |Nutrition Bars          |60g       |e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed2383

In [39]:
df_silver = df_silver.select("product_code", "division", "category", "product", "variant", "product_id", "read_timestamp", "file_name", "file_size")
df_silver.show(truncate=False)

+----------------------------------------------------------------+------------------------+-----------------+-------------------------------------------------+----------+----------+-----------------------+------------+---------+
|product_code                                                    |division                |category         |product                                          |variant   |product_id|read_timestamp         |file_name   |file_size|
+----------------------------------------------------------------+------------------------+-----------------+-------------------------------------------------+----------+----------+-----------------------+------------+---------+
|e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843|Nutrition Bars          |Energy Bars      |SportsBar Energy Bar Choco Fudge (60g)           |60g       |25891101  |2026-03-19 12:29:55.161|products.csv|1388     |
|e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e|Nutrition Bars    

### Transformed data to Silver Layer

In [40]:
if not spark.catalog.tableExists(SILVER_PRODUCTS):
    logger.info("Silver products table not found. Creating new table...")
    df_silver.write.format("delta") \
                   .option("delta.enableChangeDataFeed", "true") \
                   .option("mergeSchema", "true") \
                   .mode("overwrite") \
                   .saveAsTable(SILVER_PRODUCTS)
    logger.info("Silver products table created successfully")
else:
    logger.info("Silver products table exists. Performing upsert...")
    upsert(spark=spark, df=df_silver, key_cols=["product_id"],
           table="dim_products", cdc="read_timestamp",
           name_catalog=app.catalog_name, name_schema="silver", logger=logger)
    logger.info("Upsert completed successfully")

2026-03-19 23:18:57 | INFO     | Silver products table exists. Performing upsert...
2026-03-19 23:18:57 | INFO     | Starting UPSERT into fmcg_domain.silver.dim_products
2026-03-19 23:19:05 | INFO     | UPSERT completed successfully: fmcg_domain.silver.dim_products
2026-03-19 23:19:05 | INFO     | Upsert completed successfully


<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Gold</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Gold
    </h1>
</div>


In [41]:
# Take req cols only
# "product_code, product_id, division, category, product, variant, channel"
df_gold = df_silver.select("product_code", "product_id", "division", "category", "product", "variant")
df_gold = process_timestamp(df=df_gold)
df_gold.show(truncate=False)

+----------------------------------------------------------------+----------+------------------------+-----------------+-------------------------------------------------+----------+--------------------------+
|product_code                                                    |product_id|division                |category         |product                                          |variant   |process_timestamp         |
+----------------------------------------------------------------+----------+------------------------+-----------------+-------------------------------------------------+----------+--------------------------+
|e91ba9d665f90254da5809bfdebe3db2be01a52f50b6fd96b57eed238392b843|25891101  |Nutrition Bars          |Energy Bars      |SportsBar Energy Bar Choco Fudge (60g)           |60g       |2026-03-19 16:18:59.793698|
|e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e|25891102  |Nutrition Bars          |Energy Bars      |SportsBar Energy Bar Choco Fudge (40g)      

In [42]:
logger.info("Gold products table not found. Creating new table...")
df_gold.write.format("delta") \
                .option("delta.enableChangeDataFeed", "true") \
                .option("mergeSchema", "true") \
                .mode("overwrite") \
                .saveAsTable(GOLD_SB_PRODUCTS)
logger.info("Gold products table created successfully")

2026-03-19 23:19:05 | INFO     | Gold products table not found. Creating new table...
2026-03-19 23:19:08 | INFO     | Gold products table created successfully


## Merging Data source with parent

In [43]:
df_gold = spark.sql(f"SELECT * FROM {app.catalog_name}.gold.dim_products")
df_gold.count()

415

In [44]:
# Get the target Delta table from the metastore
delta_table = DeltaTable.forName(spark, GOLD_PRODUCTS)

# Read source data and select only required columns
df_child_products = spark.table(GOLD_SB_PRODUCTS).select(
    "product_code",
    "division",
    "category",
    "product",
    "variant"
)

# Perform MERGE (UPSERT) operation
delta_table.alias("target").merge(
    # Source DataFrame
    source=df_child_products.alias("source"),
    # Match condition between target and source
    condition="target.product_code = source.product_code"

# If the record already exists → update columns
).whenMatchedUpdate(
    set={
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }

# If the record does not exist → insert new row
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }

# Execute the merge transaction
).execute()

,num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,18,18,0,0


In [45]:
df_gold = spark.sql(f"SELECT * FROM {app.catalog_name}.gold.dim_products")
df_gold.count()

415